# Pan-African Soybean G×E Challenge — Submission Notebook (v6)

**Competition:** DataTour 2026 · dataafriquehub.org  
**Task:** predict soybean grain yield (kg/ha) in environments never seen in training  
**Metric:** RMSE  

---

## What this notebook does differently

Earlier attempts added features and models and each one scored **worse** on the
leaderboard. The diagnostics below explain why, and drive every choice here.

### Diagnostic 1 — how much signal is actually available

| quantity | value |
|---|---|
| predict the global mean for every row | RMSE 1083 |
| between-environment std | 917 |
| within-environment std | 603 |
| environment mean predicted from climate + soil alone | RMSE 809, r = 0.47 |

The environment mean drives **83%** of the variation in yield, but climate and
soil rasters barely predict it. That caps what any model can do here, and it is
why scores sit close to the 1083 constant-prediction baseline.

### Diagnostic 2 — the honest validation set

Plain `GroupKFold` OOF is **optimistic**: a validation environment can sit at a
location that also appears in the training folds under a different year. The real
test set has 56% completely unseen locations. Scoring only the OOF rows whose
location is absent from their training fold gives **RMSE 1013**, which matches
the actual leaderboard score of 1018 almost exactly. That subset — not the plain
OOF number — is the metric this notebook optimises.

### Diagnostic 3 — the ensemble under-disperses

Averaging over folds and seeds shrinks prediction variance. On unseen-location
rows the predictions are far too tightly clustered around the mean, and that
under-confidence costs real RMSE. Rescaling predictions away from the mean by a
calibrated factor α recovers it.

### What was tried and **rejected** (each hurt, measured not guessed)

| idea | result |
|---|---|
| XGBoost in the blend | 1009 OOF, dragged the ensemble down |
| log-transform of the target | skew 0.50 → −1.02, strictly worse |
| two-stage env-mean + deviation model | 1025 OOF vs 956 |
| variety-panel fingerprint features | 960.8 vs 959.6 |
| Finlay-Wilkinson / CV stability features | no gain |
| label-encoding `variety_id` to a float | **bug** — made LightGBM read 365 varieties as an ordered number |

### Final recipe

CatBoost with **native categorical handling**, seed-bagged over 4 seeds and
blended across 3 depth/l2 settings, then rescaled by α = 1.15 calibrated on
unseen-location rows.

Every seed is fixed, so re-running reproduces the submission byte for byte.

## 1. Setup

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from catboost import CatBoostRegressor, Pool
warnings.filterwarnings('ignore')

SEED, N_FOLDS, K_NBR = 42, 5, 5
SEEDS   = [42, 202, 777, 1337]              # seed bagging
CONFIGS = {(8, 3): 0.80, (7, 3): 0.10, (8, 6): 0.10}   # depth, l2 -> blend weight
ALPHA   = 1.15                              # dispersion correction (see markdown)
np.random.seed(SEED)

_c = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if _c:
    TRAIN_PATH  = _c[0]
    TEST_PATH   = TRAIN_PATH.replace('train.csv', 'test.csv')
    SAMPLE_PATH = TRAIN_PATH.replace('train.csv', 'sample_submission.csv')
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = 'train.csv', 'test.csv', 'sample_submission.csv'

TARGET, ID, ENV, VAR = 'yield_kg_ha', 'id', 'environment_id', 'variety_id'
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
print('train:', TRAIN_PATH)

## 2. Load data and fix the fold split once

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
gmean = train[TARGET].mean()

# One fixed fold split reused everywhere, so encodings and models never disagree
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(train, groups=train[ENV]))

print(f'train {train.shape} | test {test.shape}')
print(f'environments  train {train[ENV].nunique()} | test {test[ENV].nunique()} | '
      f'overlap {len(set(train[ENV]) & set(test[ENV]))}')
print(f'global mean {gmean:.0f} | target std {train[TARGET].std():.0f}')

em = train.groupby(ENV)[TARGET].mean()
print(f'between-env std {em.std():.0f} | '
      f'within-env std {(train[TARGET] - train[ENV].map(em)).std():.0f}')

## 3. Static features

In [ ]:
def static(df):
    df = df.copy()
    dt = pd.to_datetime(df['SOWING'], dayfirst=True, errors='coerce')
    df['sow_month']   = dt.dt.month.fillna(6).astype(float)
    df['sow_doy']     = dt.dt.dayofyear.fillna(180).astype(float)
    df['sow_doy_sin'] = np.sin(2 * np.pi * df['sow_doy'] / 365.25)
    df['sow_doy_cos'] = np.cos(2 * np.pi * df['sow_doy'] / 365.25)
    df['abs_lat']     = df['LAT'].abs()
    df['hemisphere']  = np.where(df['LAT'] >= 0, 'N', 'S')
    df['is_irrigated'] = (~df['RAINFED'].astype(str).str.lower()
                            .str.contains('rainfed', na=False)).astype(int)
    df['temp_annual_mean']   = df['wc2.1_30s_bio_1']
    df['precip_annual']      = df['wc2.1_30s_bio_12']
    df['precip_seasonality'] = df['wc2.1_30s_bio_15']
    df['temp_range']         = df['wc2.1_30s_bio_5'] - df['wc2.1_30s_bio_6']
    df['aridity_proxy']      = df['precip_annual'] / (df['temp_annual_mean'] + 10)
    for v in ['clay','sand','silt','nitrogen','phh2o','soc','bdod','cfvo','ocd']:
        cols = [c for c in df.columns if c.startswith(v + '_')]
        if cols:
            df[f'{v}_mean_depth'] = df[cols].mean(axis=1)
    return df

train, test = static(train), static(test)
print('static features done')

## 4. Leak-free target encodings

Variety and location statistics are computed **out-of-fold** on train (a fold
never sees its own environments) and from the full train set for test, since no
test environment appears in train.

In [ ]:
def oof_encode(df, key, aggs):
    for c in aggs:
        df[c] = np.nan
    for fit, val in FOLDS:
        st = df.iloc[fit].groupby(key)[TARGET].agg(**aggs)
        for c in st.columns:
            df.loc[df.index[val], c] = df.iloc[val][key].map(st[c]).values
    return df

VAGG = dict(variety_mean_yield='mean', variety_median_yield='median',
            variety_std_yield='std',  variety_n_trials='count')
train = oof_encode(train, VAR, VAGG)
vmed  = train['variety_std_yield'].median()
train['variety_mean_yield']   = train['variety_mean_yield'].fillna(gmean)
train['variety_median_yield'] = train['variety_median_yield'].fillna(gmean)
train['variety_std_yield']    = train['variety_std_yield'].fillna(vmed)
train['variety_n_trials']     = train['variety_n_trials'].fillna(1)

test = test.merge(train.groupby(VAR)[TARGET].agg(**VAGG), on=VAR, how='left')
test['variety_mean_yield']   = test['variety_mean_yield'].fillna(gmean)
test['variety_median_yield'] = test['variety_median_yield'].fillna(gmean)
test['variety_std_yield']    = test['variety_std_yield'].fillna(vmed)
test['variety_n_trials']     = test['variety_n_trials'].fillna(0)

LAGG = dict(loc_mean_yield='mean', loc_n_trials='count')
train = oof_encode(train, 'loc', LAGG)
train['loc_mean_yield'] = train['loc_mean_yield'].fillna(gmean)
train['loc_n_trials']   = train['loc_n_trials'].fillna(0)

lfull = train.groupby('loc').agg(loc_mean_yield=(TARGET, 'mean'),
                                 loc_n_trials=(TARGET, 'count'),
                                 loc_lat=('LAT', 'first'), loc_lon=('LON', 'first'))
test = test.merge(lfull[['loc_mean_yield', 'loc_n_trials']], on='loc', how='left')
test['loc_mean_yield'] = test['loc_mean_yield'].fillna(gmean)
test['loc_n_trials']   = test['loc_n_trials'].fillna(0)
print('target encodings done')

## 5. Spatial k-NN and G×E interactions

56% of test sites never appear in train, so an inverse-distance Haversine average
over the nearest known sites gives them a yield prior.

In [ ]:
def haversine(a1, o1, a2, o2):
    a1, o1, a2, o2 = map(np.radians, [a1, o1, a2, o2])
    h = np.sin((a2 - a1) / 2) ** 2 + np.cos(a1) * np.cos(a2) * np.sin((o2 - o1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

known = lfull.reset_index()

def knn_yield(lat, lon, exclude=None):
    d = haversine(lat, lon, known['loc_lat'].values, known['loc_lon'].values)
    if exclude is not None:
        d = np.where(known['loc'].values == exclude, np.inf, d)
    d = np.where(d == 0, 1e-3, d)
    i = np.argsort(d)[:K_NBR]
    return np.average(known['loc_mean_yield'].values[i], weights=1 / d[i])

# training rows exclude their own site, mirroring an unseen location at test time
train['spatial_knn_yield'] = [knn_yield(r.LAT, r.LON, r.loc) for r in train.itertuples()]
test['spatial_knn_yield']  = [knn_yield(r.LAT, r.LON)        for r in test.itertuples()]

for df in (train, test):
    df['gxe_variety_temp']    = df['variety_mean_yield'] * df['temp_annual_mean']
    df['gxe_variety_precip']  = df['variety_mean_yield'] * df['precip_annual']
    df['gxe_variety_aridity'] = df['variety_mean_yield'] * df['aridity_proxy']
print('spatial + G×E features done')

## 6. Categoricals kept as categoricals

This is the bug that cost the most in an earlier version: `variety_id` was
label-encoded to a float, so the model read 365 varieties as an *ordered number*
and split on meaningless thresholds. They stay categorical here, with categories
shared between train and test so the codes line up.

In [ ]:
CATS = [c for c in ['COUNTRY','SEASON','RAINFED','SOURCE','COMPANY','hemisphere', VAR]
        if c in train.columns]
DROP  = [ID, TARGET, ENV, 'SOWING', 'loc']
feats = [c for c in train.columns if c not in DROP]

X, Xt = train[feats].copy(), test[feats].copy()
for c in CATS:
    X[c], Xt[c] = X[c].astype(str), Xt[c].astype(str)

cat_idx = [feats.index(c) for c in CATS]
y       = train[TARGET].values
print(f'{len(feats)} features | {len(CATS)} categorical -> {CATS}')

## 7. Seed-bagged CatBoost blend

Three depth/l2 settings, four seeds each, all under the fixed `GroupKFold` split.
Weights come from an OOF search; depth 8 / l2 3 was the strongest single setting
across seeds (955.4), the other two add diversity.

In [ ]:
oof_tot  = np.zeros(len(y))
test_tot = np.zeros(len(Xt))

for (depth, l2), w in CONFIGS.items():
    oof_c, test_c = np.zeros(len(y)), np.zeros(len(Xt))
    for s in SEEDS:
        o = np.zeros(len(y))
        for fit, val in FOLDS:
            m = CatBoostRegressor(
                iterations=4000, learning_rate=0.02, depth=depth, l2_leaf_reg=l2,
                loss_function='RMSE', eval_metric='RMSE', random_seed=s,
                verbose=0, early_stopping_rounds=200,
                cat_features=cat_idx, one_hot_max_size=8)
            m.fit(Pool(X.iloc[fit], y[fit], cat_features=cat_idx),
                  eval_set=Pool(X.iloc[val], y[val], cat_features=cat_idx),
                  use_best_model=True)
            o[val]  = m.predict(X.iloc[val])
            test_c += m.predict(Xt) / (len(FOLDS) * len(SEEDS))
        oof_c += o / len(SEEDS)
    print(f'  depth={depth} l2={l2} bagged OOF {rmse(y, oof_c):.1f}  (weight {w})')
    oof_tot  += w * oof_c
    test_tot += w * test_c

print(f'\nblended OOF (all rows): {rmse(y, oof_tot):.1f} kg/ha')

## 8. Score on unseen locations — the honest metric

Plain OOF flatters the model. Restricting to rows whose location is absent from
their own training fold reproduces the leaderboard closely and is what α is
tuned against.

In [ ]:
newloc = np.zeros(len(y), dtype=bool)
for fit, val in FOLDS:
    fit_locs   = set(train.iloc[fit]['loc'])
    newloc[val] = ~train.iloc[val]['loc'].isin(fit_locs).values

print(f'unseen-location rows: {newloc.sum()} / {len(y)} ({100 * newloc.mean():.0f}%)')
print(f'  RMSE seen locations   : {rmse(y[~newloc], oof_tot[~newloc]):.1f}')
print(f'  RMSE UNSEEN locations : {rmse(y[newloc],  oof_tot[newloc]):.1f}   <- tracks the leaderboard')

print('\nalpha sweep   all rows | unseen locations')
for a in [1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.30]:
    all_r = rmse(y, gmean + a * (oof_tot - gmean))
    new_r = rmse(y[newloc], gmean + a * (oof_tot[newloc] - gmean))
    mark  = '  <- chosen' if abs(a - ALPHA) < 1e-9 else ''
    print(f'  alpha={a:.2f}   {all_r:8.1f} | {new_r:8.1f}{mark}')

print(f'\nalpha={ALPHA} is the only setting that improves BOTH regimes; '
      f'beyond ~1.20 seen-location rows start to degrade.')

## 9. Write the submission

In [ ]:
final = np.clip(gmean + ALPHA * (test_tot - gmean), 0, None)

sample = pd.read_csv(SAMPLE_PATH)
tcol   = [c for c in sample.columns if c != ID][0]
sub    = sample[[ID]].copy()
sub[tcol] = sub[ID].map(dict(zip(test[ID], final)))

assert sub[tcol].isna().sum() == 0, 'missing predictions'
assert len(sub) == len(sample),     'row count differs from sample_submission'
assert list(sub.columns) == list(sample.columns), 'column layout differs'

sub.to_csv('submission.csv', index=False)
print(f'submission.csv written — {len(sub)} rows')
print(f'preds  min {final.min():.0f} | mean {final.mean():.0f} | '
      f'max {final.max():.0f} | std {final.std():.0f}')
print(sub.head().to_string(index=False))

---

## Where the remaining error lives

The environment mean is 83% of the variance and climate plus soil explain little
of it (r = 0.47). Until an environment-quality signal exists that those rasters
do not carry — in-season weather for the actual trial year, management intensity,
fertiliser, or trial-level metadata — scores stay near the 1083 constant
baseline. Extra model capacity does not touch this; it is a data limit, and it is
the most likely thing separating the top of the leaderboard from the rest.

Worth trying next, in rough order of expected value:

1. **In-season weather** for each trial year rather than 30-year WorldClim
   normals — rainfall and heat stress during the growing window, not the
   long-run average. This is the single biggest gap.
2. **Yield-gap or productivity rasters** as an environment-quality prior.
3. **Days from sowing to a climatological season start**, rather than raw day of
   year, so planting date is expressed relative to the local season.
4. **Maturity group / variety pedigree**, if obtainable, to make `variety_id`
   generalise to varieties with few trials.